In [1]:
import sys
sys.path.append("../src")

In [2]:
from datachecker.batch_sources import CsvBatchSource, BatchSource, JsonlBatchSource
from datachecker.schema_loader import YamlFileSchemaLoader, SchemaSpec, SchemaLoader
from datachecker.plans import PydanticPlanCompiler, PydanticPlan
from datachecker.batch_validators import ValidationPlan, RowError, ValidationReport, BatchValidator, PydanticBatchValidator


In [3]:

from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Protocol, Sequence

from pydantic import BaseModel, ValidationError


# ---- Example "engine" that validates whole files in batches ----
def validate_source_in_batches(
    source: Any,  # BatchSource (Protocol); kept Any so this snippet is standalone
    plan: PydanticPlan,
    validator: BatchValidator,
    batch_size: int = 1000,
) -> ValidationReport:
    """
    Validates all records from a source in batches and aggregates into one report.
    Assumes the source has read_batches(batch_size=...).
    """
    all_errors: list[RowError] = []
    total = valid = invalid = 0

    for batch_no, batch in enumerate(source.read_batches(batch_size=batch_size), start=1):
        report = validator.validate_batch(plan, [dict(r) for r in batch])

        total += report.total
        valid += report.valid
        invalid += report.invalid

        # shift row_index to be global (optional, but super useful)
        base = (batch_no - 1) * batch_size
        for re in report.row_errors:
            all_errors.append(
                RowError(
                    row_index=base + re.row_index,
                    record_hint=re.record_hint,
                    errors=re.errors,
                )
            )

    return ValidationReport(
        schema_name=plan.name,
        total=total,
        valid=valid,
        invalid=invalid,
        row_errors=all_errors,
    )

In [5]:

loader = YamlFileSchemaLoader(root_dir="../schemas")
spec = loader.load("user")
plan = PydanticPlanCompiler().compile(spec)

src = JsonlBatchSource("../data/users.jsonl")
validator = PydanticBatchValidator(record_hint_keys=("id", "email"), max_errors=50)

final_report = validate_source_in_batches(src, plan, validator, batch_size=500)

print("OK?", final_report.ok())
print("Total:", final_report.total, "Valid:", final_report.valid, "Invalid:", final_report.invalid)

# show first few errors
for err in final_report.row_errors[:3]:
    print("Row:", err.row_index, "hint:", err.record_hint)
    for e in err.errors:
        print("  ", e["loc"], "-", e["msg"])

OK? True
Total: 2000 Valid: 2000 Invalid: 0
